# Attention Mechanism

Part of **ML Math Applied**.

## 1. Intuition First

Attention compares each query to all keys, normalizes those comparisons, and uses the resulting weights to average the values.

![Attention pipeline](../assets/diagrams/attention_pipeline.svg)

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
np.set_printoptions(precision=5, suppress=True)
torch.set_printoptions(precision=5)

## 2. Derivation

For query matrix $Q \in \mathbb{R}^{n_q \times d_k}$, key matrix $K \in \mathbb{R}^{n_k \times d_k}$, and value matrix $V \in \mathbb{R}^{n_k \times d_v}$:

$$
S = \frac{QK^\top}{\sqrt{d_k}}
$$

gives pairwise similarity scores. Applying a row-wise softmax turns each query row into a probability distribution:

$$
A = \mathrm{softmax}(S).
$$

The final context vectors are then

$$
C = AV.
$$

In causal language modeling, a lower-triangular mask sets all future-token scores to $-\infty$ before softmax.

In [ ]:
from src.ml_components import attention_scores, causal_attention_mask, scaled_dot_product_attention

tokens = ["the", "robot", "solves", "math"]
Q = np.array([[[1.0, 0.2, 0.0], [0.9, 0.8, 0.1], [0.2, 1.0, 0.8], [0.1, 0.7, 1.2]]])
K = np.array([[[1.1, 0.1, 0.1], [0.7, 0.9, 0.0], [0.3, 0.9, 0.7], [0.1, 0.8, 1.1]]])
V = np.array([[[1.0, 0.0], [0.7, 0.4], [0.2, 1.0], [0.1, 1.3]]])

causal_mask = causal_attention_mask(len(tokens))[None, :, :]
raw_scores = attention_scores(Q, K)
context, weights = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

print("masked weights =\n", weights[0])
print("last-token context =", context[0, -1])

## 3. PyTorch Verification

The same masked attention computation in PyTorch should agree elementwise.

In [ ]:
Q_t = torch.tensor(Q, dtype=torch.float64)
K_t = torch.tensor(K, dtype=torch.float64)
V_t = torch.tensor(V, dtype=torch.float64)
mask_t = torch.tensor(causal_mask)

scores_t = Q_t @ K_t.transpose(-1, -2) / np.sqrt(Q.shape[-1])
scores_t = torch.where(mask_t, scores_t, torch.full_like(scores_t, -1e9))
context_t = torch.softmax(scores_t, dim=-1) @ V_t

print(torch.allclose(context_t, torch.tensor(context), atol=1e-8))

## 4. Custom Figure

The heatmap shows who can attend to whom after the causal mask is applied.

In [ ]:
entropy = -(weights[0] * np.log(np.clip(weights[0], 1e-12, None))).sum(axis=-1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
im = axes[0].imshow(weights[0], cmap="YlOrBr", vmin=0.0, vmax=1.0)
axes[0].set_xticks(range(len(tokens)), tokens)
axes[0].set_yticks(range(len(tokens)), tokens)
axes[0].set_title("Causal attention weights")
for row in range(len(tokens)):
    for col in range(len(tokens)):
        axes[0].text(col, row, f"{weights[0, row, col]:.2f}", ha="center", va="center", fontsize=9)

axes[1].plot(tokens, entropy, marker="o", linewidth=2.5, color="tab:blue")
axes[1].set_ylim(0.0, max(entropy) + 0.15)
axes[1].set_title("Row entropy after masking")
axes[1].set_ylabel("nats")
axes[1].grid(True, alpha=0.3)

fig.colorbar(im, ax=axes[0], fraction=0.046)
plt.tight_layout()
plt.show()

## 5. Case Study: Next-Token Prediction

The last token can aggregate information from all previous positions, but the first token can only attend to itself.
That asymmetry is exactly what autoregressive decoding needs.